# Entrenamiento AKOrN con C_scale en Google Colab

Este notebook permite entrenar modelos AKOrN con diferentes valores de `c_scale` para estudiar la transición de fase en las dinámicas de Kuramoto.

## Experimentos a ejecutar

Entrenaremos con 8 valores de C_scale (los restantes del sweep):
- 0.0434, 0.0806, 0.1178, 0.155, 0.1922, 0.2294, 0.2666, 0.3038

**Configuración optimizada:**
- Dataset: MNIST
- n=2, ch=64, T=10, L=2
- Batch size: 128, Epochs: 15
- Checkpoints en epochs: 5, 10, 15

## 1. Verificar GPU y Configurar Entorno

In [ ]:
# Verificar GPU disponible
import torch
print(f"🔍 GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Nombre: {torch.cuda.get_device_name(0)}")
    print(f"   Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No hay GPU. Ve a Runtime -> Change runtime type -> GPU")

## 2. Clonar Repositorio e Instalar Dependencias

In [ ]:
import os
import sys

# Directorio base de Colab
base_dir = '/content'
repo_name = 'Proyecto-Inv.-teorica.'
target_path = f'{base_dir}/{repo_name}/codigo/akornA'

print(f"📍 Directorio actual: {os.getcwd()}")

# Verificar si ya estamos en el directorio correcto
if os.getcwd() == target_path and os.path.exists('requirements.txt') and os.path.exists('source'):
    print("✅ Ya estamos en el directorio correcto")
else:
    # Volver al directorio base
    os.chdir(base_dir)
    print(f"📂 En directorio: {base_dir}")
    
    # Eliminar repo anterior si existe
    if os.path.exists(repo_name):
        print("🗑️  Eliminando repo anterior...")
        !rm -rf {repo_name}
    
    # Clonar branch mac específicamente
    print("📥 Clonando repositorio (branch mac)...")
    !git clone -b mac --single-branch https://github.com/ACRCris/Proyecto-Inv.-teorica..git
    
    # Navegar al directorio target
    os.chdir(target_path)
    print(f"📂 Navegando a: {target_path}")

# Agregar al sys.path para que Python encuentre el módulo source
if target_path not in sys.path:
    sys.path.insert(0, target_path)
    print(f"✅ Agregado al PYTHONPATH: {target_path}")

# Verificar estado final
print(f"\n{'='*60}")
print(f"📍 Directorio final: {os.getcwd()}")
print(f"✅ requirements.txt: {os.path.exists('requirements.txt')}")
print(f"✅ source/: {os.path.exists('source')}")
print(f"✅ train_classification.py: {os.path.exists('train_classification.py')}")
print(f"={'='*60}\n")

# Instalar dependencias
if os.path.exists('requirements.txt'):
    print("📦 Instalando dependencias...")
    !pip install -q -r requirements.txt
    print("✅ Dependencias instaladas")
else:
    print("❌ ERROR: requirements.txt no encontrado")

# Descargar MNIST automáticamente
print("\n📥 Descargando dataset MNIST...")
import torch
from torchvision import datasets
datasets.MNIST(root='./data', train=True, download=True)
datasets.MNIST(root='./data', train=False, download=True)
print("✅ MNIST descargado")

## 3. Configurar Valores de C_scale y Parámetros

In [ ]:
# Valores de C_scale para los experimentos restantes
C_SCALE_VALUES = [0.0434, 0.0806, 0.1178, 0.155, 0.1922, 0.2294, 0.2666, 0.3038]

# Configuración del modelo (optimizada para rendimiento)
CONFIG = {
    'data': 'mnist',
    'epochs': 15,
    'batchsize': 128,
    'lr': 0.0001,
    'n': 2,
    'ch': 64,
    'T': 10,
    'L': 2,
    'ksizes': '7 5',
    'gamma': 0.7,
    'checkpoint_every': 5,
    'adveval_freq': 10,
    'device': 'auto'
}

print(f"📊 Configuración:")
print(f"   • {len(C_SCALE_VALUES)} experimentos")
print(f"   • C_scale valores: {C_SCALE_VALUES}")
print(f"   • Epochs: {CONFIG['epochs']}")
print(f"   • Batch size: {CONFIG['batchsize']}")
print(f"   • Checkpoints cada {CONFIG['checkpoint_every']} epochs")

## 4. Función para Entrenar un Experimento

Esta función ejecuta el entrenamiento para un valor específico de C_scale.

In [ ]:
import subprocess
import time
from datetime import datetime

def train_experiment(c_scale_value, config):
    """
    Entrena un modelo con un valor específico de c_scale.
    
    Args:
        c_scale_value: Valor de c_scale para el experimento
        config: Diccionario con la configuración del modelo
    """
    exp_name = f"c_scale_{c_scale_value}"
    
    print(f"\n{'='*60}")
    print(f"🚀 Iniciando experimento: {exp_name}")
    print(f"{'='*60}")
    print(f"⏱️  Inicio: {datetime.now().strftime('%H:%M:%S')}")
    
    # Construir comando
    cmd = [
        'python', 'train_classification.py', exp_name,
        '--data', config['data'],
        '--epochs', str(config['epochs']),
        '--batchsize', str(config['batchsize']),
        '--lr', str(config['lr']),
        '--n', str(config['n']),
        '--ch', str(config['ch']),
        '--T', str(config['T']),
        '--L', str(config['L']),
        '--ksizes', *config['ksizes'].split(),
        '--gamma', str(config['gamma']),
        '--c_scale', str(c_scale_value),
        '--checkpoint_every', str(config['checkpoint_every']),
        '--adveval_freq', str(config['adveval_freq']),
        '--device', config['device']
    ]
    
    # Configurar PYTHONPATH para que subprocess encuentre el módulo source
    import os
    env = os.environ.copy()
    env['PYTHONPATH'] = os.getcwd()
    
    # Ejecutar entrenamiento con captura de salida
    start_time = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    elapsed_time = time.time() - start_time
    
    if result.returncode == 0:
        print(f"\n✅ Experimento completado exitosamente")
        print(f"⏱️  Tiempo total: {elapsed_time/3600:.2f} horas")
        print(f"📁 Checkpoints guardados en: runs/{exp_name}/")
    else:
        print(f"\n❌ Error en el experimento (código: {result.returncode})")
        print("\n📄 STDOUT:")
        print(result.stdout)
        print("\n📄 STDERR:")
        print(result.stderr)
    
    return result.returncode == 0

print("✅ Función de entrenamiento definida")

## 5. Ejecutar Todos los Experimentos

**Importante:** Cada experimento toma aproximadamente 2-3 horas en GPU T4 de Colab.

Puedes ejecutar todos de una vez o seleccionar experimentos individuales modificando la lista.

In [ ]:
# Ejecutar todos los experimentos secuencialmente
results = {}
total_start = time.time()

print(f"🎯 Iniciando sweep de {len(C_SCALE_VALUES)} experimentos")
print(f"⏱️  Estimado: ~{len(C_SCALE_VALUES) * 2.5:.1f} horas\n")

for i, c_scale in enumerate(C_SCALE_VALUES, 1):
    print(f"\n{'#'*60}")
    print(f"# Experimento {i}/{len(C_SCALE_VALUES)}: C_scale = {c_scale}")
    print(f"{'#'*60}\n")
    
    success = train_experiment(c_scale, CONFIG)
    results[c_scale] = 'SUCCESS' if success else 'FAILED'
    
    # Mostrar progreso
    elapsed = (time.time() - total_start) / 3600
    remaining = len(C_SCALE_VALUES) - i
    print(f"\n📊 Progreso general: {i}/{len(C_SCALE_VALUES)} ({i/len(C_SCALE_VALUES)*100:.1f}%)")
    print(f"⏱️  Tiempo transcurrido: {elapsed:.2f}h")
    if i < len(C_SCALE_VALUES):
        print(f"⏳ Estimado restante: ~{remaining * 2.5:.1f}h")

# Resumen final
total_time = (time.time() - total_start) / 3600
print(f"\n{'='*60}")
print(f"✅ SWEEP COMPLETADO")
print(f"{'='*60}")
print(f"⏱️  Tiempo total: {total_time:.2f} horas")
print(f"\n📊 Resultados:")
for c_scale, status in results.items():
    icon = "✅" if status == "SUCCESS" else "❌"
    print(f"   {icon} C_scale {c_scale}: {status}")

## 6. (Opcional) Ejecutar un Solo Experimento

Si prefieres ejecutar un experimento específico en lugar de todos:

In [ ]:
# Descomenta y modifica para ejecutar un experimento individual
# c_scale_single = 0.0806  # Cambia este valor
# train_experiment(c_scale_single, CONFIG)

In [ ]:
# TEST DIRECTO: Ejecutar comando manualmente para ver error completo
import subprocess
import os

cmd = [
    'python', 'train_classification.py', 'test_debug',
    '--data', 'mnist',
    '--epochs', '1',
    '--batchsize', '128',
    '--lr', '0.0001',
    '--n', '2',
    '--ch', '64',
    '--T', '10',
    '--L', '2',
    '--ksizes', '7', '5',
    '--gamma', '0.7',
    '--c_scale', '0.0434',
    '--checkpoint_every', '5',
    '--adveval_freq', '10',
    '--device', 'auto'
]

print("🔍 Ejecutando comando:")
print(" ".join(cmd))
print("\n" + "="*60)

# Configurar PYTHONPATH para subprocess
env = os.environ.copy()
env['PYTHONPATH'] = os.getcwd()

result = subprocess.run(cmd, capture_output=True, text=True, env=env)

print(f"\n🔴 Return code: {result.returncode}")
print("\n📤 STDOUT:")
print(result.stdout if result.stdout else "(vacío)")
print("\n📤 STDERR:")
print(result.stderr if result.stderr else "(vacío)")
print("="*60)

## 7. Verificar Checkpoints Generados

In [ ]:
import os
from pathlib import Path

# Listar todos los experimentos completados
runs_dir = Path('runs')
print("📁 Experimentos completados:\n")

for c_scale in C_SCALE_VALUES:
    exp_dir = runs_dir / f"c_scale_{c_scale}"
    if exp_dir.exists():
        # Buscar checkpoints
        checkpoints = list(exp_dir.glob("checkpoint_*.pth"))
        ema_checkpoints = list(exp_dir.glob("ema_*.pth"))
        
        print(f"✅ c_scale_{c_scale}:")
        print(f"   • Checkpoints: {len(checkpoints)}")
        print(f"   • EMA checkpoints: {len(ema_checkpoints)}")
        
        # Mostrar tamaño total
        total_size = sum(f.stat().st_size for f in exp_dir.rglob("*.pth"))
        print(f"   • Tamaño total: {total_size / 1e6:.1f} MB")
    else:
        print(f"❌ c_scale_{c_scale}: No encontrado")
    print()

## 8. Comprimir y Descargar Resultados

Comprime todos los checkpoints para descargarlos a tu máquina local.

In [ ]:
# Comprimir todos los experimentos
!tar -czf c_critico_experiments.tar.gz runs/c_scale_*

# Verificar tamaño del archivo comprimido
import os
size_mb = os.path.getsize('c_critico_experiments.tar.gz') / 1e6
print(f"📦 Archivo comprimido: c_critico_experiments.tar.gz ({size_mb:.1f} MB)")

# Descargar archivo
from google.colab import files
print("\n⬇️  Descargando archivo...")
files.download('c_critico_experiments.tar.gz')
print("✅ Descarga iniciada")

## 9. (Opcional) Monitorear Progreso Durante Entrenamiento

Si quieres monitorear el progreso mientras entrena, ejecuta esta celda en paralelo:

In [ ]:
# Monitorear logs de TensorBoard
%load_ext tensorboard
%tensorboard --logdir runs/

# O ver archivos de log directamente
# !tail -f runs/c_scale_0.0434/train.log 2>/dev/null || echo "Aún no hay logs"

## 📝 Notas Importantes

1. **Tiempo de ejecución**: Cada experimento toma ~2-3 horas en GPU T4 de Colab. Los 8 experimentos tomarán aproximadamente 16-24 horas.

2. **Sesión de Colab**: Colab puede desconectarse después de 12 horas. Para evitar perder progreso:
   - Ejecuta experimentos en bloques más pequeños
   - O usa Colab Pro para sesiones más largas

3. **Checkpoints**: Se guardan cada 5 epochs (epochs 5, 10, 15) para cada experimento.

4. **Memoria**: Cada experimento genera ~50-100 MB de checkpoints.

5. **Continuación**: Si la sesión se interrumpe, puedes reejecutar solo los experimentos faltantes modificando `C_SCALE_VALUES`.